# Exemplo de Execução do Pipeline (CNN 2D)

Este notebook demonstra o uso passo a passo dos componentes do pipeline de Machine Learning.

In [3]:
import os
import sys
import torch
import numpy as np
from sklearn.model_selection import train_test_split

# Em notebooks, __file__ não existe por padrão. Usamos o diretório atual para encontrar a raiz do projeto.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from ai.loader.loader import DataLoader
from ai.label.label_generator import LabelGenerator
from ai.preprocess.cnn2d import PreprocessCNN2D
from ai.models.cnn2d import ModelCNN2D
from ai.trainer.trainer import ModelTrainer
from ai.evaluation.monitor import ModelMonitor
from ai.evaluation.summary import ModelSummary

## 1. Carregamento dos Dados

Vamos usar a classe `DataLoader` para buscar os arquivos parquet e carregá-los em um DataFrame do Pandas.

In [4]:
# Ajuste o data_path para apontar para seus arquivos reais de dados.
# Se não passar nada, o loader buscará por padrão na pasta data/parquet/
data_path = os.path.join(PROJECT_ROOT, "data", "parquet", "**", "*.parquet")

loader = DataLoader(data_path=data_path, max_files=1) # Limitando a 1 arquivo para o exemplo rodar rápido
df = loader.execute()

# Geração de Labels
if df is not None:
    df = LabelGenerator.apply_label(df, file_path_col='file_path', label_col='has_truth_clus')
    df.drop(columns=['file_path'], inplace=True)

print(f"Tamanho do DataFrame: {len(df) if df is not None else 0}")

Encontrados 4 arquivos válidos


Carregando Parquets: 100%|██████████| 4/4 [00:00<00:00, 30.99arquivo/s]

   run_number  event_number  avgmu  cl_idx         cl_et  cl_eta  cl_phi  \
0  1268421252    1082130432    0.0       0  36421.988281  -2.276   1.190   
1  1268421252    1084227584    0.0       0  18038.785156   2.000   1.976   
2  1268421252    1118306304    0.0       0  36231.109375   1.412  -2.663   
3  1268421252    1118437376    0.0       0  34674.773438  -2.001  -1.779   
4  1268421252    1118568448    0.0       0  48019.136719   0.662   2.246   

    cl_rhad   cl_rphi  cl_eratio  ...  cl_truth_ring_91  cl_truth_ring_92  \
0  0.052155  0.591873   0.257673  ...              -1.0              -1.0   
1  0.047198  0.706759   0.444810  ...              -1.0              -1.0   
2  0.020149  0.596413   0.577704  ...              -1.0              -1.0   
3  0.088464  0.920760   0.325904  ...              -1.0              -1.0   
4  0.019916  0.933692   0.511916  ...              -1.0              -1.0   

   cl_truth_ring_93  cl_truth_ring_94  cl_truth_ring_95  cl_truth_ring_96  \
0  

## 2. Pré-processamento

O pré-processador formata as variáveis em tensores adequados para a CNN (imagens 2D com múltiplos canais).

In [5]:
preprocessor = PreprocessCNN2D()

if df is not None:
    X = preprocessor.transform(df)
    Y = preprocessor.get_labels(df, label_col='has_truth_clus')
    
    print(f"Formato de X (Features): {X.shape}")
    print(f"Formato de Y (Labels): {Y.shape}")
else:
    print("Sem dados para pré-processar.")


Convertendo camadas do calorímetro em Imagens 2D (Tensores)...
[1/7] Processando canal: cl_cells_presampler


Processando Amostras: 100%|██████████| 7583/7583 [00:00<00:00, 25611.41it/s]


[2/7] Processando canal: cl_cells_em1


Processando Amostras: 100%|██████████| 7583/7583 [00:00<00:00, 23752.22it/s]


[3/7] Processando canal: cl_cells_em2


...

[6/7] Processando canal: cl_cells_had2


Processando Amostras: 100%|██████████| 7583/7583 [00:00<00:00, 28254.07it/s]


[7/7] Processando canal: cl_cells_had3


Processando Amostras: 100%|██████████| 7583/7583 [00:00<00:00, 27959.22it/s]

Formato de X (Features): (7583, 7, 7, 15)
Formato de Y (Labels): (7583,)


## 3. Divisão de Dados (Train / Test)

Separamos um conjunto de teste isolado.

In [6]:
if df is not None:
    # Vamos separar 15% para o teste isolado que NUNCA vai ser visto no treinamento.
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.15, random_state=42)
    print(f"Treinamento: {X_train.shape[0]} amostras")
    print(f"Teste Isolado: {X_test.shape[0]} amostras")

Treinamento: 6445 amostras
Teste Isolado: 1138 amostras


## 4. Treinamento

Configuramos o treinador (`ModelTrainer`) que lidará com o ciclo de vida do PyTorch Lightning. Podemos usar Holdout Simples ou K-Fold.

In [7]:
if df is not None:
    # Configurações do treinamento
    results_dir = os.path.join(PROJECT_ROOT, "results", "CNN2D")
    
    trainer = ModelTrainer(
        max_epochs=2, # Poucas épocas só para testar o código
        batch_size=32,
        patience=3,
        log_dir=os.path.join(results_dir, "lightning_logs")
    )
    
    # Treino Simples (Holdout)
    print("\nIniciando treinamento simples...")
    model = ModelCNN2D(learning_rate=0.001)
    trained_trainer = trainer.fit(model, X_train, Y_train)

/home/joao.gomes/LPS/cern/cern/lib/python3.13/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/joao.gomes/LPS/cern/cern/lib/python3.13/site-p ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
/home/joao.gomes/LPS/cern/cern/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging


Iniciando treinamento simples...
Preparando DataLoaders para Holdout...
Iniciando treinamento do modelo ModelCNN2D...
Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/home/joao.gomes/LPS/cern/cern/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/joao.gomes/LPS/cern/cern/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/home/joao.gomes/LPS/cern/cern/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 0:   9%|▊         | 14/162 [00:00<00:01, 104.37it/s, v_num=2, train_loss_step=0.184]

/home/joao.gomes/LPS/cern/cern/lib/python3.13/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No negative samples in targets, false positive value should be meaningless. Returning zero tensor in false positive score
  warnings.warn(*args, **kwargs)


Epoch 0: 100%|██████████| 162/162 [00:01<00:00, 110.30it/s, v_num=2, train_loss_step=0.0173, val_loss=0.140, val_acc=0.960, val_auc=0.852, train_loss_epoch=0.179, train_acc=0.955]

Metric val_loss improved. New best score: 0.140


Epoch 1: 100%|██████████| 162/162 [00:01<00:00, 123.25it/s, v_num=2, train_loss_step=1.090, val_loss=0.121, val_acc=0.960, val_auc=0.880, train_loss_epoch=0.136, train_acc=0.957] 

Metric val_loss improved by 0.019 >= min_delta = 0.0. New best score: 0.121
`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|██████████| 162/162 [00:01<00:00, 116.52it/s, v_num=2, train_loss_step=1.090, val_loss=0.121, val_acc=0.960, val_auc=0.880, train_loss_epoch=0.136, train_acc=0.957]
Treinamento finalizado! Melhor modelo: /home/joao.gomes/LPS/cern/results/CNN2D/lightning_logs/ModelCNN2D-epoch=01-val_loss=0.1214.ckpt


### Exemplo alternativo usando K-Fold (Opcional)
Se quiséssemos rodar validação cruzada, seria assim:

In [8]:
if df is not None:
    print("\n(Demonstração) Iniciando K-Fold...")
    # Comentei para não rodar e demorar no exemplo.
    # model_kwargs = {'learning_rate': 0.001}
    # fold_trainers, fold_models = trainer.fit_kfold(ModelCNN2D, model_kwargs, X_train, Y_train, n_splits=3)


(Demonstração) Iniciando K-Fold...


## 5. Avaliação

Vamos avaliar o modelo treinado em nosso Test Set (dados isolados).

In [9]:
if df is not None:
    monitor = ModelMonitor(output_dir=os.path.join(results_dir, "plots"))
    summary = ModelSummary(output_dir=os.path.join(results_dir, "metrics"))

    # Ativa modo de inferência
    model.eval()
    
    with torch.no_grad():
        X_tensor = torch.as_tensor(X_test, dtype=torch.float32)
        logits = model(X_tensor)
        y_prob = torch.sigmoid(logits).numpy().flatten()
        
    y_true = Y_test.flatten()
    threshold = 0.5
    y_pred = (y_prob >= threshold).astype(int)
    
    # Salva CSV
    summary.save_metrics(y_true, y_prob, threshold=threshold, filename="notebook_test_metrics.csv")
    
    # Plota gráficos
    monitor.plot_roc_curve(y_true, y_prob, filename="notebook_roc_curve.pdf")
    monitor.plot_confusion_matrix(y_true, y_pred, filename="notebook_confusion_matrix.pdf")
    
    print("Avaliação finalizada!")

Métricas anexadas ao arquivo: /home/joao.gomes/LPS/cern/results/CNN2D/metrics/notebook_test_metrics.csv
Curva ROC salva em: /home/joao.gomes/LPS/cern/results/CNN2D/plots/notebook_roc_curve.pdf
Matriz de Confusão salva em: /home/joao.gomes/LPS/cern/results/CNN2D/plots/notebook_confusion_matrix.pdf
Avaliação finalizada!
